In [1]:
import os

for root, dirs, files in os.walk("/kaggle/input"):
    print(root)

/kaggle/input
/kaggle/input/datasets
/kaggle/input/datasets/organizations
/kaggle/input/datasets/organizations/snap
/kaggle/input/datasets/organizations/snap/amazon-fine-food-reviews
/kaggle/input/notebooks
/kaggle/input/notebooks/ivoluisorellana
/kaggle/input/notebooks/ivoluisorellana/6a-finetuning-t5
/kaggle/input/notebooks/ivoluisorellana/6a-finetuning-t5/t5_finetuned_etapa5
/kaggle/input/notebooks/ivoluisorellana/6a-finetuning-t5/t5_finetuned_etapa5/checkpoint-24859
/kaggle/input/notebooks/ivoluisorellana/6a-finetuning-t5/__results___files
/kaggle/input/notebooks/ivoluisorellana/6a-finetuning-t5/t5_finetuned_deploy
/kaggle/input/notebooks/ivoluisorellana/6a-finetuning-t5/t5_finetuned_deploy/model
/kaggle/input/notebooks/ivoluisorellana/6a-finetuning-t5/t5_finetuned_final


In [2]:
import os

base_path = "/kaggle/input/notebooks/ivoluisorellana/6a-finetuning-t5/__results___files"

for root, dirs, files in os.walk(base_path):
    print("CARPETA:", root)
    if files:
        print("ARCHIVOS:", files[:20])
    print("-" * 80)

CARPETA: /kaggle/input/notebooks/ivoluisorellana/6a-finetuning-t5/__results___files
ARCHIVOS: ['__results___4_0.png']
--------------------------------------------------------------------------------


In [3]:
# NOTEBOOK 6b - Carga del modelo fine-tuned para demo Gradio

!pip install -q transformers sentencepiece accelerate gradio

import os
import json
import torch
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

print("CUDA disponible:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU detectada:", torch.cuda.get_device_name(0))
    print("CUDA capability:", torch.cuda.get_device_capability(0))
else:
    print("ATENCION: no hay GPU activa")

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# --------------------------------------------------
# RUTA REAL (YA CORREGIDA)
# --------------------------------------------------
deploy_root = "/kaggle/input/notebooks/ivoluisorellana/6a-finetuning-t5/t5_finetuned_deploy"

model_path = os.path.join(deploy_root, "model")
config_path = os.path.join(deploy_root, "inference_config.json")
examples_path = os.path.join(deploy_root, "demo_examples.json")

print("Ruta deploy_root:", deploy_root)
print("Ruta model_path:", model_path)
print("Existe deploy_root?:", os.path.exists(deploy_root))
print("Existe model_path?:", os.path.exists(model_path))
print("Existe config_path?:", os.path.exists(config_path))
print("Existe examples_path?:", os.path.exists(examples_path))

if not os.path.exists(deploy_root):
    raise FileNotFoundError(f"No se encontro la carpeta: {deploy_root}")

if not os.path.exists(model_path):
    raise FileNotFoundError(f"No se encontro el modelo: {model_path}")

with open(config_path, "r") as f:
    inference_config = json.load(f)

with open(examples_path, "r") as f:
    demo_examples = json.load(f)

tokenizer_gradio = AutoTokenizer.from_pretrained(model_path)
model_gradio = AutoModelForSeq2SeqLM.from_pretrained(model_path).to(device)

print("\nModelo cargado correctamente desde:")
print(model_path)

print("\nConfiguracion de inferencia:")
print(inference_config)

CUDA disponible: False
ATENCION: no hay GPU activa
Ruta deploy_root: /kaggle/input/notebooks/ivoluisorellana/6a-finetuning-t5/t5_finetuned_deploy
Ruta model_path: /kaggle/input/notebooks/ivoluisorellana/6a-finetuning-t5/t5_finetuned_deploy/model
Existe deploy_root?: True
Existe model_path?: True
Existe config_path?: True
Existe examples_path?: True


Loading weights:   0%|          | 0/131 [00:00<?, ?it/s]


Modelo cargado correctamente desde:
/kaggle/input/notebooks/ivoluisorellana/6a-finetuning-t5/t5_finetuned_deploy/model

Configuracion de inferencia:
{'base_model': 't5-small', 'task_prefix': 'summarize: ', 'max_input_length': 256, 'max_output_length': 32, 'num_beams': 4, 'early_stopping': True}


In [4]:
# Funcion de inferencia para resumir reviews

def resumir_review(texto):
    texto = str(texto).strip()

    if len(texto) == 0:
        return "Por favor, ingresa una review."

    entrada = inference_config["task_prefix"] + texto

    inputs = tokenizer_gradio(
        entrada,
        return_tensors="pt",
        truncation=True,
        padding=True,
        max_length=inference_config["max_input_length"]
    ).to(device)

    with torch.no_grad():
        generated_ids = model_gradio.generate(
            **inputs,
            max_length=inference_config["max_output_length"],
            num_beams=inference_config["num_beams"],
            early_stopping=inference_config["early_stopping"]
        )

    resumen = tokenizer_gradio.batch_decode(
        generated_ids,
        skip_special_tokens=True
    )[0]

    return resumen.strip()

# Prueba rapida
review_prueba = """
this product arrived on time, tastes excellent, and the packaging was very good.
i would definitely buy it again because the flavor is rich and fresh.
"""

print("Review de prueba:")
print(review_prueba)

print("\nResumen generado:")
print(resumir_review(review_prueba))

Review de prueba:

this product arrived on time, tastes excellent, and the packaging was very good.
i would definitely buy it again because the flavor is rich and fresh.


Resumen generado:
great product


In [5]:
# Despliegue final con Gradio

import gradio as gr

descripcion = """
Este demo permite ingresar una review de Amazon Fine Food Reviews en ingles
y generar automaticamente un resumen usando el modelo T5 fine-tuned.
El notebook 6b solo carga el modelo ya entrenado desde los artefactos exportados por 6a.
"""

ejemplos = [[x] for x in demo_examples]

interfaz = gr.Interface(
    fn=resumir_review,
    inputs=gr.Textbox(
        lines=8,
        placeholder="Write an English review here..."
    ),
    outputs=gr.Textbox(
        lines=3,
        label="Generated summary"
    ),
    title="Amazon Review Summarizer - T5 Fine-Tuned",
    description=descripcion,
    examples=ejemplos
)

interfaz.launch(share=True)

* Running on local URL:  http://127.0.0.1:7860
* Running on public URL: https://9437afbd6cd78b8d83.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
